# Instalando bibliotecas e dependencias

In [ ]:
## Bloco de codigo para instalar versao especifica dos pacotes
#!pip install pyspark

In [ ]:
#from pyspark.sql.functions import input_file_name, split, col, lit, regexp_replace
#from datetime import datetime, timedelta
#from decimal import Decimal
#from pyspark.sql.functions import col, count, when, isnull, isnan, countDistinct, round, variance, stddev
#from pyspark.sql.types import NumericType, StringType
#from pyspark.sql import functions as F
import os
import time
from datetime import datetime

# Inicializando o Spark

In [ ]:
from pyspark.sql import SparkSession
# Inicializando a sessão
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Camada Silver") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

# Verificando se funcionou
print("Sessão Spark criada com sucesso!")
spark

Sessão Spark criada com sucesso!


# Instanciando o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Criar uma função log para registrar a data de importação dos dados

In [ ]:
#Criar uma função log para registrar a data de importação dos dados
def log():
    return datetime.now().strftime('%Y/%m/%d-%H:%M:%S')
dt_proc = log()
#current_date = datetime.now().strftime('%d/%m/%Y')

dt_proc

'2026/01/09-23:13:17'

# Leitura de um arquivo para testar a conexão

In [ ]:
path_file = '/content/drive/MyDrive/hackathon_pod_2025/database/raw/book_atraso/dados_faturamento/part-00000-4fcc9763-9f46-4a0b-9e86-7f5b7789bfb8-c000.snappy.parquet'

df_atraso = spark.read.parquet (path_file, header=True, inferSchema=True)
df_atraso.createOrReplaceTempView("df_atraso_um_parquet")

df_atraso.show()

+-----------+------------------+--------------------+------------------+---------+-------------+------------------------+--------------+-------+--------+---------------------+---------+---------------------+---------------------+-------------------+------------------------+-------------------+--------------+--------------------------+----------------------------+--------------------+---------------------+----------------------+------------------+------------------+------------------+----------------------+----------------+------------------+-------------------+------+-------+--------+-------+----------------+----------+---------------+-------------+---------------+--------------+----------------+-----------------------+--------------+------------------+---------------+----------------------+---------------------+-----------------+----------------------+------------------+
|    NUM_CPF|    DAT_REFERENCIA|     NUM_FATURA_HASH|NUM_ENT_SEQ_FATURA| CONTRATO|DW_UN_NEGOCIO|DW_HIS_PONTO_VENDA_

#Leitura de todos os arquivo parquet e registro da view df_pagamentos para o spark SQL

In [ ]:
target_directory = '/content/drive/MyDrive/hackathon_pod_2025/database/raw/book_atraso/dados_faturamento/'

parquet_files = [os.path.join(target_directory,f) for f in os.listdir(target_directory) if f.endswith('.parquet')]

if not parquet_files:
    print(f"No .parquet files found in the directory: {target_directory}")
else:
  print(f"Loading {len(parquet_files)} parquet files from: {target_directory}")
  df_atraso= spark.read.parquet(*parquet_files, header=True, inferSchema=True)
  df_atraso.createOrReplaceTempView("df_atraso")
  df_atraso.show()

Loading 10 parquet files from: /content/drive/MyDrive/hackathon_pod_2025/database/raw/book_atraso/dados_faturamento/
+-----------+------------------+--------------------+------------------+---------+-------------+------------------------+--------------+-------+--------+---------------------+---------+---------------------+---------------------+-------------------+------------------------+-------------------+--------------+--------------------------+----------------------------+--------------------+---------------------+----------------------+------------------+------------------+------------------+----------------------+----------------+------------------+-------------------+------+-------+--------+-------+----------------+----------+---------------+-------------+---------------+--------------+----------------+-----------------------+--------------+------------------+---------------+----------------------+---------------------+-----------------+----------------------+------------------

In [ ]:
#Quantidade de linhas
df_atraso.count()

31611316

In [ ]:
#Estrutura dos dados
df_atraso.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- DAT_REFERENCIA: string (nullable = true)
 |-- NUM_FATURA_HASH: string (nullable = true)
 |-- NUM_ENT_SEQ_FATURA: string (nullable = true)
 |-- CONTRATO: string (nullable = true)
 |-- DW_UN_NEGOCIO: string (nullable = true)
 |-- DW_HIS_PONTO_VENDA_COMTA: string (nullable = true)
 |-- DW_NUM_CLIENTE: string (nullable = true)
 |-- DW_AREA: string (nullable = true)
 |-- DW_CICLO: string (nullable = true)
 |-- DW_TIPO_CLIENTE_CONTA: string (nullable = true)
 |-- DW_OFERTA: string (nullable = true)
 |-- DW_FAIXA_AGING_FATURA: string (nullable = true)
 |-- DW_FAIXA_AGING_DIVIDA: string (nullable = true)
 |-- DW_FAIXA_TEMPO_BASE: string (nullable = true)
 |-- DW_FAIXA_AGING_PROX_FECH: string (nullable = true)
 |-- DW_TIPO_FATURAMENTO: string (nullable = true)
 |-- COD_PLATAFORMA: string (nullable = true)
 |-- DAT_CRIACAO_REGISTRO_TRANS: string (nullable = true)
 |-- DAT_ALTERACAO_REGISTRO_TRANS: string (nullable = true)
 |-- DAT_CANCELAMENTO_FAT

##Testando se temos diferença nas horas

In [ ]:
#testando os campos para identificar se temos diferença nas horas
spark.sql("""
select distinct
  --substr(DAT_REFERENCIA, -8, 10) as DAT_STATUS_FATURA
  --substr(DAT_ORIGINAL_VCTO_FAT,-8,10) as DAT_ORIGINAL_VCTO_FAT
  --substr(DAT_ALTERACAO_VCTO_FAT,-8,10)  AS DAT_ALTERACAO_VCTO_FAT
  --substr(DAT_CRIACAO_FAT,-8,10)  AS DAT_CRIACAO_FAT
  substr(DAT_VENCIMENTO_FAT,-8,10)  AS DAT_VENCIMENTO_FAT
from df_atraso
""").show()

+------------------+
|DAT_VENCIMENTO_FAT|
+------------------+
|          00:00:00|
+------------------+



#Ajustando os campos com os tipos corretos

In [ ]:
dt_proc = log()

df_atraso_ajustado = spark.sql(f"""
SELECT
'{dt_proc}' as DATA_IMPORT,
    CAST(NUM_CPF AS STRING)                                     AS NUM_CPF,
    TO_DATE(DAT_REFERENCIA, 'ddMMMyyyy:HH:mm:ss')               AS DAT_REFERENCIA,
    CAST(NUM_FATURA_HASH AS STRING)                              AS NUM_FATURA_HASH,
    CAST(NUM_ENT_SEQ_FATURA AS INT)                              AS NUM_ENT_SEQ_FATURA,
    CAST(CONTRATO AS BIGINT)                                     AS CONTRATO,
    CAST(DW_UN_NEGOCIO AS INT)                                   AS DW_UN_NEGOCIO,
    CAST(DW_HIS_PONTO_VENDA_COMTA AS BIGINT)                     AS DW_HIS_PONTO_VENDA_COMTA,
    CAST(DW_NUM_CLIENTE AS BIGINT)                               AS DW_NUM_CLIENTE,
    CAST(DW_AREA AS INT)                                         AS DW_AREA,
    CAST(DW_CICLO AS INT)                                        AS DW_CICLO,
    CAST(DW_TIPO_CLIENTE_CONTA AS INT)                           AS DW_TIPO_CLIENTE_CONTA,
    CAST(DW_OFERTA AS INT)                                       AS DW_OFERTA,
    CAST(DW_FAIXA_AGING_FATURA AS INT)                           AS DW_FAIXA_AGING_FATURA,
    CAST(DW_FAIXA_AGING_DIVIDA AS INT)                           AS DW_FAIXA_AGING_DIVIDA,
    CAST(DW_FAIXA_TEMPO_BASE AS INT)                             AS DW_FAIXA_TEMPO_BASE,
    CAST(DW_FAIXA_AGING_PROX_FECH AS INT)                        AS DW_FAIXA_AGING_PROX_FECH,
    CAST(DW_TIPO_FATURAMENTO AS INT)                             AS DW_TIPO_FATURAMENTO,
    CAST(COD_PLATAFORMA AS STRING)                               AS COD_PLATAFORMA,

    TO_DATE(DAT_CRIACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss')    AS DAT_CRIACAO_REGISTRO_TRANS,
    TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')    AS HR_CRIACAO_REGISTRO_TRANS,
    TO_DATE(DAT_ALTERACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss')                           AS DAT_ALTERACAO_REGISTRO_TRANS,
    TO_CHAR(TO_TIMESTAMP(DAT_ALTERACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')  AS HR_ALTERACAO_REGISTRO_TRANS,
    TO_DATE(DAT_CANCELAMENTO_FAT, 'ddMMMyyyy:HH:mm:ss')         AS DAT_CANCELAMENTO_FAT,
    TO_DATE(DAT_ORIGINAL_VCTO_FAT,'ddMMMyyyy:HH:mm:ss')         AS DAT_ORIGINAL_VCTO_FAT,
    TO_DATE(DAT_ALTERACAO_VCTO_FAT,'ddMMMyyyy:HH:mm:ss')        AS DAT_ALTERACAO_VCTO_FAT,
    TO_DATE(DAT_CRIACAO_FAT,'ddMMMyyyy:HH:mm:ss')               AS DAT_CRIACAO_FAT,
    TO_DATE(DAT_VENCIMENTO_FAT,'ddMMMyyyy:HH:mm:ss')            AS DAT_VENCIMENTO_FAT,
    TO_DATE(DAT_STATUS_FAT,'ddMMMyyyy:HH:mm:ss')                AS DAT_STATUS_FAT,
    TO_CHAR(TO_TIMESTAMP(DAT_STATUS_FAT,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')                AS HR_STATUS_FAT,
    TO_DATE(DAT_MIN_VENCIMENTO_FAT,'ddMMMyyyy:HH:mm:ss')                                 AS DAT_MIN_VENCIMENTO_FAT,
    TO_CHAR(TO_TIMESTAMP(DAT_MIN_VENCIMENTO_FAT,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')        AS HR_MIN_VENCIMENTO_FAT,

    CAST(NUM_BILL_SEQ_FAT AS INT)                                AS NUM_BILL_SEQ_FAT,
    CAST(NUM_SEQ_ACORDO_FAT AS INT)                              AS NUM_SEQ_ACORDO_FAT,

    CAST(IND_ISENCAO_COB_FAT AS STRING)                          AS IND_ISENCAO_COB_FAT,
    CAST(IND_WO AS STRING)                                       AS IND_WO,
    CAST(IND_PDD AS STRING)                                      AS IND_PDD,
    CAST(IND_PCCR AS STRING)                                     AS IND_PCCR,
    CAST(IND_ACA AS STRING)                                      AS IND_ACA,
    CAST(IND_PRIMEIRA_FAT AS STRING)                             AS IND_PRIMEIRA_FAT,
    CAST(IND_FRAUDE AS STRING)                                   AS IND_FRAUDE,

    CAST(VAL_FAT_LIQUIDO AS DECIMAL(18,2))                       AS VAL_FAT_LIQUIDO,
    CAST(VAL_FAT_BRUTO AS DECIMAL(18,2))                         AS VAL_FAT_BRUTO,
    CAST(VAL_FAT_CREDITO AS DECIMAL(18,2))                       AS VAL_FAT_CREDITO,
    CAST(VAL_FAT_AJUSTE AS DECIMAL(18,2))                        AS VAL_FAT_AJUSTE,
    CAST(VAL_FAT_BRUTO_BC AS DECIMAL(18,2))                      AS VAL_FAT_BRUTO_BC,
    CAST(VAL_FAT_PAGAMENTO_BRUTO AS DECIMAL(18,2))               AS VAL_FAT_PAGAMENTO_BRUTO,
    CAST(VAL_FAT_ABERTO AS DECIMAL(18,2))                        AS VAL_FAT_ABERTO,
    CAST(VAL_FAT_ABERTO_LIQ AS DECIMAL(18,2))                    AS VAL_FAT_ABERTO_LIQ,
    CAST(VAL_MULTA_JUROS AS DECIMAL(18,2))                       AS VAL_MULTA_JUROS,
    CAST(VAL_MULTA_CANCELAMENTO AS DECIMAL(18,2))               AS VAL_MULTA_CANCELAMENTO,
    CAST(VAL_PARC_APARELHO_LIQ AS DECIMAL(18,2))                 AS VAL_PARC_APARELHO_LIQ,
    CAST(VAL_FAT_LIQ_JM_MC AS DECIMAL(18,2))                     AS VAL_FAT_LIQ_JM_MC,

    TO_DATE(DAT_ATIVACAO_CONTA_CLI,'ddMMMyyyy:HH:mm:ss')    AS DAT_ATIVACAO_CONTA_CLI,
    TO_CHAR(TO_TIMESTAMP(DAT_ATIVACAO_CONTA_CLI,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')    AS HR_ATIVACAO_CONTA_CLI,
    TO_DATE(DAT_CRIACAO_DW,'ddMMMyyyy:HH:mm:ss')            AS DAT_CRIACAO_DW,
    TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_DW,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')            AS HR_CRIACAO_DW
FROM df_atraso
""")
df_atraso_ajustado.createOrReplaceTempView("df_atraso_ajustado")
df_atraso_ajustado.show()


+-------------------+-----------+--------------+--------------------+------------------+---------+-------------+------------------------+--------------+-------+--------+---------------------+---------+---------------------+---------------------+-------------------+------------------------+-------------------+--------------+--------------------------+-------------------------+----------------------------+---------------------------+--------------------+---------------------+----------------------+---------------+------------------+--------------+-------------+----------------------+---------------------+----------------+------------------+-------------------+------+-------+--------+-------+----------------+----------+---------------+-------------+---------------+--------------+----------------+-----------------------+--------------+------------------+---------------+----------------------+---------------------+-----------------+----------------------+---------------------+--------------

#Salvar o arquivo na camada Silver

In [ ]:
#Avaliar como faremos a partição
  #.partitionBy("DATA_IMPORT") \

target_directory_write = "/content/drive/MyDrive/hackathon_pod_2025/database/silver/book_atraso/dados_faturamento/"

df_atraso_ajustado.write \
  .mode("append") \
  .parquet(target_directory_write)